### Data Ingestion and Preprocessing

In [1]:
import pandas as pd
import numpy as np
from langchain_core.documents import Document
# two tower models

In [2]:
import torch
print(torch.cuda.is_available())        # True
print(torch.cuda.get_device_name(0))    # e.g. "NVIDIA RTX 3080"
print(torch.cuda.mem_get_info())

True
NVIDIA GeForce GTX 1650
(3454795776, 4294639616)


In [3]:
questions_df = pd.read_parquet("C:\\Users\\satya\\Desktop\\Stack_Overflow_AI_Assistant\\data\\questions_cleaned.parquet")
top_tags_df  = pd.read_parquet("C:\\Users\\satya\\Desktop\\Stack_Overflow_AI_Assistant\\data\\top_tags.parquet")

In [4]:
questions_df.columns

Index(['id', 'title', 'body', 'code_blocks', 'tags', 'primary_tag', 'score',
       'view_count', 'answer_count', 'has_accepted_answer', 'creation_date',
       'title_length', 'body_length', 'body_word_count', 'has_code',
       'num_tags'],
      dtype='str')

In [5]:
top_tags_df.columns

Index(['tag_name', 'question_count'], dtype='str')

In [7]:
tag_lookup = dict(zip(top_tags_df["tag_name"], top_tags_df["question_count"]))

In [8]:
# Question and tag ingestion as langchain documents

def ingest_questions(questions_df, tag_lookup):
    documents = []
    
    for _, row in questions_df.iterrows():
        #page_content combining the title and body of the question
        page_content = f"[TITLE] {row['title']}\n\n[BODY] {row['body']}"
        #metadata storing the question attributes and tag popularity
        metadata = {
            "question_id": int(row["id"]),
            "tags": str(row["tags"]),
            "title": str(row["title"]),
            "primary_tag": str(row["primary_tag"]),

            "score": int(row["score"]),
            "view_count": int(row["view_count"]),
            "answer_count": int(row["answer_count"]),
            "has_accepted_answer": bool(row["has_accepted_answer"]),

            "has_code": bool(row["has_code"]),
            "num_tags": int(row["num_tags"]),
            "title_length": int(row["title_length"]),
            "body_length": int(row["body_length"]),
            "body_word_count": int(row["body_word_count"]),

            "creation_date": str(row["creation_date"]),
            "tag_popularity": int(tag_lookup.get(row["primary_tag"], 0))
        }

        documents.append(Document(page_content=page_content, metadata=metadata))

    return documents


In [9]:
print("\nIngesting questions...")
question_docs = ingest_questions(questions_df, tag_lookup)


Ingesting questions...


In [10]:
print(f"Total documents created: {len(question_docs):,}")

Total documents created: 498,643


In [11]:
sample = question_docs[0]
print(f"\nSample page_content:\n{sample.page_content[:200]}...") # Checking sanity of ingestion by printing the first 200 characters of the page_content of the first document
print(f"\nSample metadata:\n{sample.metadata}")


Sample page_content:
[TITLE] Why is processing a sorted array faster than processing an unsorted array?

[BODY] Here is a piece of C++ code that shows some very peculiar behavior. For some strange reason, sorting the data...

Sample metadata:
{'question_id': 11227809, 'tags': 'java,c++,performance,cpu-architecture,branch-prediction', 'title': 'Why is processing a sorted array faster than processing an unsorted array?', 'primary_tag': 'java', 'score': 26621, 'view_count': 1746329, 'answer_count': 27, 'has_accepted_answer': True, 'has_code': True, 'num_tags': 5, 'title_length': 74, 'body_length': 1216, 'body_word_count': 205, 'creation_date': '2012-06-27 13:51:36.160000+00:00', 'tag_popularity': 1866055}


### Embedding and VectorStore DB

In [13]:
from sentence_transformers import SentenceTransformer
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
# Creating an embedding manager class to handle seamless embedding generation and storage using SentenceTransformer
class EmbeddingManager:
    '''Manages embedding generation and storage using SentenceTransformer'''

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self._load_model()

    def _load_model(self):
        '''Loads the SentenceTransformer model'''
        try:
            self.model = SentenceTransformer(self.model_name, device=self.device)
            print(f"Loaded embedding model: {self.model_name}")
            print(f"Model device: {self.device}")
            if self.device == "cuda":
                print(f"GPU: {torch.cuda.get_device_name(0)}")
                print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
        except Exception as e:
            print(f"Error loading model: {e}")
            raise
    
    def generate_embeddings(self, documents: List[Document]) -> np.ndarray:
        """Generates embeddings from a list of LangChain Documents"""

        if not self.model:
            raise ValueError("Model not loaded successfully.")
        
        # Extract page_content from each Document
        texts = [doc.page_content for doc in documents]

        print(f"\nGenerating embeddings for {len(texts):,} documents on {self.device}...")
        batch_size = 512 if self.device == "cuda" else 32
        embeddings = self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
            device=self.device,
            convert_to_numpy=True
        )
        print(f"Generated embeddings with shape: {embeddings.shape}")
        if self.device == "cuda":
            torch.cuda.empty_cache()
            print(f"GPU cache cleared")
        return embeddings


In [15]:
embedding_manager = EmbeddingManager()
question_embeddings = embedding_manager.generate_embeddings(question_docs)
print(f"Question embeddings shape: {question_embeddings.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11462.28it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded embedding model: all-MiniLM-L6-v2
Model device: cuda
GPU: NVIDIA GeForce GTX 1650
GPU Memory: 4.3 GB

Generating embeddings for 498,643 documents on cuda...


Batches: 100%|██████████| 974/974 [28:59<00:00,  1.79s/it]


Generated embeddings with shape: (498643, 384)
GPU cache cleared
Question embeddings shape: (498643, 384)


In [41]:
def check_semantic_similarity_manual(query, embedding_manager, embeddings, questions_df, top_k=5):
    # Encode manual text query
    query_vec = embedding_manager.model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0].reshape(1, -1)

    # Compare against all stored question embeddings
    scores = cosine_similarity(query_vec, embeddings)[0]
    top_idx = np.argsort(scores)[::-1][:top_k]

    print(f"Query: {query}\n")
    print("Top Similar Questions:")
    for i, idx in enumerate(top_idx, start=1):
        title = questions_df.iloc[idx]["title"]
        tag = questions_df.iloc[idx]["primary_tag"]
        print(f"{i}. [{scores[idx]:.3f}] {title} ({tag})")

    return [
        {
            "rank": i + 1,
            "score": float(scores[idx]),
            "index": int(idx),
            "title": questions_df.iloc[idx]["title"],
            "primary_tag": questions_df.iloc[idx]["primary_tag"]
        }
        for i, idx in enumerate(top_idx)
    ]

In [44]:
# Manual query examples
results = check_semantic_similarity_manual(
    query="how to write a pythone code?",
    embedding_manager=embedding_manager,
    embeddings=question_embeddings,
    questions_df=questions_df,
    top_k=5
)

# results_2 = check_semantic_similarity_manual(
#     query="difference between list and tuple",
#     embedding_manager=embedding_manager,
#     embeddings=question_embeddings,
#     questions_df=questions_df,
#     top_k=5
# )

Query: how to write a pythone code?

Top Similar Questions:
1. [0.702] How to create a programming language in Python (python)
2. [0.701] Beginner wondering if his code is 'Pythonic' (python)
3. [0.632] Whats the best way to write python code into a python file? (python)
4. [0.626] how to convert Python 3 to Python 2 code? (python)
5. [0.621] Beginner Python Practice? (python)


In [17]:
def evaluate_accuracy(embeddings, questions_df, sample_size=1000):
    '''Evaluates embedding accuracy using 3 metrics'''

    print("=" * 50)
    print("      EMBEDDING ACCURACY REPORT")
    print("=" * 50)

    sample_indices = np.random.choice(len(questions_df), sample_size, replace=False)

    # ─── 1. Same Tag Accuracy ─────────────────────────────────────────────────
    # Top 1 nearest neighbor should be same tag as query
    print(f"\n📊 1. Same-Tag Accuracy (top-1, top-3, top-5)")
    top1_correct = 0
    top3_correct = 0
    top5_correct = 0

    for idx in sample_indices:
        query_vec  = embeddings[idx].reshape(1, -1)
        scores     = cosine_similarity(query_vec, embeddings)[0]
        top5_idx   = np.argsort(scores)[::-1][1:6]       # exclude itself

        query_tag  = questions_df.iloc[idx]["primary_tag"]
        top5_tags  = [questions_df.iloc[i]["primary_tag"] for i in top5_idx]

        if top5_tags[0] == query_tag:                     top1_correct += 1
        if query_tag in top5_tags[:3]:                    top3_correct += 1
        if query_tag in top5_tags[:5]:                    top5_correct += 1

    print(f"   Top-1 Accuracy : {top1_correct / sample_size * 100:.1f}%")
    print(f"   Top-3 Accuracy : {top3_correct / sample_size * 100:.1f}%")
    print(f"   Top-5 Accuracy : {top5_correct / sample_size * 100:.1f}%")

    # ─── 2. Score-based Accuracy ──────────────────────────────────────────────
    # Higher scored questions should rank higher in similar results
    print(f"\n📊 2. Score-based Ranking Accuracy")
    score_correct = 0

    for idx in sample_indices:
        query_vec   = embeddings[idx].reshape(1, -1)
        scores      = cosine_similarity(query_vec, embeddings)[0]
        top5_idx    = np.argsort(scores)[::-1][1:6]

        query_tag   = questions_df.iloc[idx]["primary_tag"]
        same_tag    = [i for i in top5_idx if questions_df.iloc[i]["primary_tag"] == query_tag]

        if len(same_tag) >= 2:
            # Check if higher similarity score → higher question score
            sim_scores      = [scores[i] for i in same_tag[:2]]
            question_scores = [questions_df.iloc[i]["score"] for i in same_tag[:2]]
            if (sim_scores[0] > sim_scores[1]) == (question_scores[0] > question_scores[1]):
                score_correct += 1

    print(f"   Score Ranking Accuracy : {score_correct / sample_size * 100:.1f}%")

    # ─── 3. Overall Accuracy ──────────────────────────────────────────────────
    overall = (top1_correct + top3_correct + top5_correct) / (sample_size * 3) * 100
    print(f"\n📊 3. Overall Accuracy Score : {overall:.1f}%")

    # ─── Interpretation ───────────────────────────────────────────────────────
    print(f"\n📊 Interpretation")
    thresholds = [
        (80, "🟢 Excellent — embeddings are highly accurate"),
        (60, "🟡 Good     — acceptable for recommendations"),
        (40, "🟠 Fair     — consider longer text or better model"),
        (0,  "🔴 Poor     — embeddings need improvement"),
    ]
    for threshold, label in thresholds:
        if overall >= threshold:
            print(f"   {label}")
            break

    print("=" * 50)
    return {
        "top1_accuracy"          : round(top1_correct / sample_size * 100, 1),
        "top3_accuracy"          : round(top3_correct / sample_size * 100, 1),
        "top5_accuracy"          : round(top5_correct / sample_size * 100, 1),
        "score_ranking_accuracy" : round(score_correct / sample_size * 100, 1),
        "overall_accuracy"       : round(overall, 1)
    }

# ─── Run Evaluation ───────────────────────────────────────────────────────────
results = evaluate_accuracy(question_embeddings, questions_df, sample_size=1000)

      EMBEDDING ACCURACY REPORT

📊 1. Same-Tag Accuracy (top-1, top-3, top-5)
   Top-1 Accuracy : 65.1%
   Top-3 Accuracy : 84.7%
   Top-5 Accuracy : 90.0%

📊 2. Score-based Ranking Accuracy
   Score Ranking Accuracy : 38.3%

📊 3. Overall Accuracy Score : 79.9%

📊 Interpretation
   🟡 Good     — acceptable for recommendations


In [ ]:
import faiss
import os
import pickle

class VectorStore:
    '''Manages vector storage and retrieval using FAISS with GPU support'''

    def __init__(self, 
                 index_name: str = "questions",
                 persist_directory: str = "../data/vector_store"):
        
        self.index_name        = index_name
        self.persist_directory = persist_directory
        self.index             = None
        self.documents         = []
        self.use_gpu           = faiss.get_num_gpus() > 0
        
        os.makedirs(self.persist_directory, exist_ok=True)
        print(f"Vector store initialized")
        print(f"FAISS GPU available : {self.use_gpu}")
        print(f"Persist directory  : {self.persist_directory}")

    def _build_index(self, embeddings: np.ndarray):
        '''Builds FAISS index from embeddings'''

        dimension = embeddings.shape[1]          # 384 for all-MiniLM-L6-v2

        # IVFFlat index — faster search on large datasets (500K+)
        # nlist = number of clusters (rule of thumb: sqrt of total docs)
        nlist      = int(np.sqrt(len(embeddings)))
        quantizer  = faiss.IndexFlatIP(dimension)         # Inner product (cosine if normalized)
        index      = faiss.IndexIVFFlat(quantizer, dimension, nlist, faiss.METRIC_INNER_PRODUCT)

        # Move to GPU if available
        if self.use_gpu:
            res        = faiss.StandardGpuResources()
            index      = faiss.index_cpu_to_gpu(res, 0, index)
            print(f"FAISS index moved to GPU")

        # Train index (required for IVFFlat)
        print(f"Training FAISS index on {len(embeddings):,} vectors...")
        index.train(embeddings.astype(np.float32))

        # Add embeddings with their integer IDs
        index.add_with_ids(
            embeddings.astype(np.float32),
            np.arange(len(embeddings))            # IDs = 0, 1, 2, ... n
        )

        print(f"FAISS index built — total vectors: {index.ntotal:,}")
        return index

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        '''Builds FAISS index and stores documents'''

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match")

        print(f"\nBuilding FAISS index for {len(documents):,} documents...")

        # Store documents
        self.documents = documents

        # Build FAISS index
        self.index = self._build_index(embeddings)

        print(f"Successfully added {len(documents):,} documents")
        print(f"Index total vectors: {self.index.ntotal:,}")

    def save(self):
        '''Saves FAISS index and documents locally'''

        index_path    = os.path.join(self.persist_directory, f"{self.index_name}.index")
        docs_path     = os.path.join(self.persist_directory, f"{self.index_name}_docs.pkl")

        # Move back to CPU before saving
        cpu_index = faiss.index_gpu_to_cpu(self.index) if self.use_gpu else self.index
        faiss.write_index(cpu_index, index_path)

        # Save documents
        with open(docs_path, "wb") as f:
            pickle.dump(self.documents, f)

        print(f"FAISS index saved  : {index_path}")
        print(f"Documents saved    : {docs_path}")

    def load(self):
        '''Loads FAISS index and documents from disk'''

        index_path = os.path.join(self.persist_directory, f"{self.index_name}.index")
        docs_path  = os.path.join(self.persist_directory, f"{self.index_name}_docs.pkl")

        if not os.path.exists(index_path):
            raise FileNotFoundError(f"No saved index found at {index_path}")

        # Load index
        cpu_index = faiss.read_index(index_path)

        # Move to GPU if available
        if self.use_gpu:
            res        = faiss.StandardGpuResources()
            self.index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
            print(f"✅ FAISS index loaded to GPU")
        else:
            self.index = cpu_index

        # Load documents
        with open(docs_path, "rb") as f:
            self.documents = pickle.load(f)

        print(f"Loaded index with {self.index.ntotal:,} vectors")
        print(f"Loaded {len(self.documents):,} documents")

    def search(self, query_embedding: np.ndarray, top_k: int = 5, nprobe: int = 10):
        '''Searches for similar documents given a query embedding'''

        if self.index is None:
            raise ValueError("Index not built. Run add_documents() or load() first.")

        # nprobe = number of clusters to search (higher = more accurate but slower)
        self.index.nprobe = nprobe

        query = query_embedding.reshape(1, -1).astype(np.float32)
        distances, indices = self.index.search(query, top_k)

        results = []
        for dist, idx in zip(distances[0], indices[0]):
            if idx == -1:
                continue
            results.append({
                "document" : self.documents[idx],
                "score"    : float(dist),
                "index"    : int(idx)
            })
        return results

    def get_stats(self):
        '''Prints vector store statistics'''
        print(f"\n{'=' * 40}")
        print(f"   VECTOR STORE STATS")
        print(f"{'=' * 40}")
        print(f"   Index name   : {self.index_name}")
        print(f"   Total vectors: {self.index.ntotal:,}" if self.index else "   Index        : Not built")
        print(f"   Documents    : {len(self.documents):,}")
        print(f"   GPU enabled  : {self.use_gpu}")
        print(f"   Directory    : {self.persist_directory}")
        print(f"{'=' * 40}")

# ─── Usage ────────────────────────────────────────────────────────────────────
vectorstore = VectorStore(
    index_name        = "questions",
    persist_directory = "../data/vector_store"
)

# # Add documents and embeddings
# vectorstore.add_documents(question_docs, question_embeddings)

# # Save locally — no re-embedding next time
# vectorstore.save()

# # Stats
# vectorstore.get_stats()

✅ Vector store initialized
✅ FAISS GPU available : False
✅ Persist directory  : ../data/vector_store

Building FAISS index for 498,643 documents...
Training FAISS index on 498,643 vectors...
✅ FAISS index built — total vectors: 498,643
✅ Successfully added 498,643 documents
✅ Index total vectors: 498,643
✅ FAISS index saved  : ../data/vector_store\questions.index
✅ Documents saved    : ../data/vector_store\questions_docs.pkl

   VECTOR STORE STATS
   Index name   : questions
   Total vectors: 498,643
   Documents    : 498,643
   GPU enabled  : False
   Directory    : ../data/vector_store


In [35]:
class RAGretriever:
    '''Handles query based retrieval from the vector store'''

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.5, nprobe: int = 10):
        '''Retrieve relevant documents for a query'''
        print(f"\nQuery          : '{query}'")
        print(f"Top-K          : {top_k}")
        print(f"Score threshold: {score_threshold}")
        
        # Generate embedding for the query
        query_embedding = self.embedding_manager.model.encode(
            [query],
            normalize_embeddings=True,
            convert_to_numpy=True
        )[0]

        # Search in vector store
        try:
            results = self.vector_store.search(
                query_embedding=query_embedding,
                top_k=top_k,
                nprobe=nprobe
            )

            # Filter results based on score threshold
            retrieved_docs = []
            for rank, result in enumerate(results):
                if result["score"] >= score_threshold:
                    doc = result["document"]
                    metadata = doc.metadata

                    retrieved_docs.append({
                        "rank": rank + 1,
                        "score": round(result["score"], 4),
                        "question_id": metadata.get("question_id"),
                        "title": metadata.get("title"),
                        "primary_tag": metadata.get("primary_tag"),
                        "tags": metadata.get("tags"),
                        "answer_count": metadata.get("answer_count"),
                        "has_accepted_answer": metadata.get("has_accepted_answer"),
                        "score_votes": metadata.get("score"),
                        "view_count": metadata.get("view_count"),
                        "has_code": metadata.get("has_code"),
                        "creation_date": metadata.get("creation_date"),
                        "tag_popularity": metadata.get("tag_popularity"),
                        "page_content": doc.page_content
                    })

            print(f"Retrieved {len(retrieved_docs)} documents\n")

            for doc in retrieved_docs:
                print(f"  Rank {doc['rank']} | Score: {doc['score']:.4f}")
                print(f"  Title : {doc['title']}")
                print(f"  Tag   : {doc['primary_tag']} | Answers: {doc['answer_count']} | Accepted: {doc['has_accepted_answer']}")
                print(f"  {'─' * 55}")

            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            raise

    def retrieve_by_tag(self, query: str, tag: str, top_k: int = 5) -> List[Dict[str, Any]]:
        '''Retrieves documents filtered by a specific tag'''

        results = self.retrieve(query, top_k=top_k * 3)  # fetch more, then filter

        filtered = [r for r in results if r["primary_tag"] == tag][:top_k]

        print(f"Filtered to {len(filtered)} results for tag: '{tag}'")
        return filtered

    def retrieve_answered_only(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        '''Retrieves only questions that have accepted answers'''

        results = self.retrieve(query, top_k=top_k * 3)  # fetch more, then filter

        filtered = [r for r in results if r["has_accepted_answer"]][:top_k]

        print(f"Filtered to {len(filtered)} answered questions")
        return filtered

In [46]:
rag_retriever = RAGretriever(vectorstore, embedding_manager)

In [45]:
results = rag_retriever.retrieve("how to write a pythone code?", top_k=5)


Query          : 'how to write a pythone code?'
Top-K          : 5
Score threshold: 0.5
Retrieved 5 documents

  Rank 1 | Score: 0.7018
  Title : How to create a programming language in Python
  Tag   : python | Answers: 2 | Accepted: True
  ───────────────────────────────────────────────────────
  Rank 2 | Score: 0.7008
  Title : Beginner wondering if his code is 'Pythonic'
  Tag   : python | Answers: 12 | Accepted: False
  ───────────────────────────────────────────────────────
  Rank 3 | Score: 0.6321
  Title : Whats the best way to write python code into a python file?
  Tag   : python | Answers: 4 | Accepted: True
  ───────────────────────────────────────────────────────
  Rank 4 | Score: 0.6264
  Title : how to convert Python 3 to Python 2 code?
  Tag   : python | Answers: 1 | Accepted: True
  ───────────────────────────────────────────────────────
  Rank 5 | Score: 0.6213
  Title : Beginner Python Practice?
  Tag   : python | Answers: 8 | Accepted: True
  ──────────────────────